In [30]:
!pip install langgraph langchain langchain-google-genai

In [31]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
import os

In [32]:
os.environ['GOOGLE_API_KEY']="xyz"


In [33]:
llm=ChatGoogleGenerativeAI(
    model='gemini-2.5-flash'
)

In [34]:
from typing import TypedDict

class ClinicalState(TypedDict, total=False):
    patient_input: str
    patient_history: str

    symptoms: str
    severity: str
    medical_entities: str
    distress_level: str
    history_risk: str
    emergency_flag: str

    medical_guidelines: str
    emergency_guidelines: str
    similar_cases: str
    specialist_recommendations: str
    specialist_suggestion: str
    drug_interactions: str

    routed_agent: str
    route_reason: str

    triage_summary: str
    diagnostic_tests: str
    follow_up: str
    emergency_notification: str

In [35]:
def symptom_extraction_agent(state: ClinicalState):
    prompt=f"""
    You are a symptom extraction agent.
    Extract patient symptoms from input.
    Patient Input: {state['patient_input']}
    """
    response=llm.invoke(prompt)
    return {
        "symptoms":response.content.strip().lower()
    }

In [36]:
def risk_scoring_agent(state:ClinicalState):
  prompt = f"""
    You are a Risk Scoring Agent.

    Analyze patient symptoms and classify severity:
    - low
    - moderate
    - high
    - critical

    Consider patient history if provided.

    Patient Input:
    {state['patient_input']}

    Patient History:
    {state.get('patient_history', '')}
    """
  response=llm.invoke(prompt)
  return{
      "severity":response.content.strip().lower()
  }

In [37]:
def medical_entity_recognition_agent(state: ClinicalState):

    prompt = f"""
    You are a Medical Entity Recognition Agent.

    Identify:
    - symptoms
    - diseases
    - medications
    - medical conditions

    Patient Input:
    {state['patient_input']}
    """
    response=llm.invoke(prompt)
    return{
        "medical_entities":response.content.strip().lower()
    }

In [38]:
def distress_analysis_agent(state: ClinicalState):

    prompt = f"""
    You are a Distress Analysis Agent.

    Analyze emotional distress level:
    - low
    - medium
    - high

    Patient Input:
    {state['patient_input']}
    """
    response=llm.invoke(prompt)
    return{
        "distress_level":response.content.strip().lower()
    }

In [39]:
def patient_history_analysis_agent(state: ClinicalState):

    prompt = f"""
    You are a Patient History Analysis Agent.

    Analyze the available patient history and identify whether it increases risk.
    Return one of:
    - low
    - moderate
    - high

    Patient History:
    {state.get('patient_history', 'Not provided')}
    """
    response = llm.invoke(prompt)
    return {
        "history_risk": response.content.strip().lower()
    }

In [40]:
def medical_guideline_retrieval_agent(state: ClinicalState):
    if str(state.get("emergency_flag", "false")).lower() == "true":
        return {"medical_guidelines": "skipped due to emergency escalation"}

    prompt = f"""
    You are a Medical Guideline Retrieval Agent.

    Retrieve standard medical guidelines
    for low/moderate severity cases.

    Symptoms:
    {state.get('symptoms')}

    Severity:
    {state.get('severity')}
    """
    response=llm.invoke(prompt)
    return{
        "medical_guidelines":response.content.strip().lower()
    }

In [41]:
def similar_case_retrieval_agent(state: ClinicalState):
  if str(state.get("emergency_flag", "false")).lower() == "true":
      return {"similar_cases": "skipped due to emergency escalation"}

  prompt = f"""
 You are a Similar Historical Case Retrieval Agent.
 Retrieve a similar prior patient case.

 Symptoms:
 {state.get('symptoms')}
"""
  response=llm.invoke(prompt)
  return{
      "similar_cases":response.content.strip().lower()
  }

In [42]:
def drug_interaction_retrieval_agent(state: ClinicalState):
  if str(state.get("emergency_flag", "false")).lower() == "true":
      return {"drug_interactions": "skipped due to emergency escalation"}

  prompt = f"""
You are a Drug Interaction Retrieval Agent.

Check whether the patient's symptoms could be related to medication use,
allergy, or adverse drug reaction.

Medical Entities:
{state.get('medical_entities')}
"""
  response=llm.invoke(prompt)
  return{
    "drug_interactions":response.content.strip().lower()
}

In [43]:
def specialist_recommendation_agent(state: ClinicalState):
 prompt = f"""
You are a Specialist Recommendation Agent.

Recommend the most suitable specialist department.

Patient Input:
{state['patient_input']}

Patient History:
{state.get('patient_history')}

History Risk:
{state.get('history_risk')}

Symptoms:
{state.get('symptoms')}

Severity:
{state.get('severity')}

Medical Entities:
{state.get('medical_entities')}

Relevant Medical Guidelines:
{state.get('medical_guidelines')}

Drug Interactions:
{state.get('drug_interactions')}
"""
 response=llm.invoke(prompt)
 return{
    "specialist_recommendations":response.content.strip().lower()
}

In [44]:
def severity_route(state):

    prompt = f"""
    Decide the next route.

    Rules:
    - high/critical → emergency_escalation_agent
    - low/moderate → medical_guideline_retrieval_agent

    Severity:
    {state.get('severity')}

    Return ONLY:
    - emergency_escalation_agent
    - medical_guideline_retrieval_agent
    """

    response = llm.invoke(prompt)

    return response.content.strip().lower()

In [45]:
def medication_route(state: ClinicalState):

    prompt = f"""
    Decide the next route.

    Rules:
    - medication reaction → drug_interaction_retrieval_agent
    - otherwise → similar_case_retrieval_agent

    Medical Entities:
    {state.get('medical_entities')}

    Return ONLY:
    - drug_interaction_retrieval_agent
    - similar_case_retrieval_agent
    """

    response = llm.invoke(prompt)

    return response.content.strip().lower()

In [46]:
def intelligent_routing(state):

    prompt = f"""
    You are an Intelligent Clinical Routing Agent.

    Route the patient to the correct medical department
    based on symptoms, severity, patient history, and emergency indicators.

    Possible routing:
    - Emergency symptoms → emergency_escalation_agent
    - Mild/common symptoms → general_physician_agent
    - Medication reactions → pharmacy_review_agent
    - Neurological symptoms → neurology_agent
    - Cardiac indicators → cardiology_agent

    Patient State:
    {state}

    Return ONLY one route name from:
    - emergency_escalation_agent
    - general_physician_agent
    - pharmacy_review_agent
    - neurology_agent
    - cardiology_agent
    """

    response = llm.invoke(prompt)

    return response.content.strip().lower()

In [47]:
def emergency_escalation_agent(state: ClinicalState):

    return {
        "routed_agent": "Emergency Escalation",
        "route_reason": "Emergency symptoms detected"
    }

In [48]:
def cardiology_agent(state: ClinicalState):

    return {
        "routed_agent": "Cardiology"
    }

In [49]:
def neurology_agent(state: ClinicalState):

    return {
        "routed_agent": "Neurology"
    }

In [50]:
def pharmacy_review_agent(state: ClinicalState):

    return {
        "routed_agent": "Pharmacy Review"
    }

In [51]:
def general_physician_agent(state: ClinicalState):

    return {
        "routed_agent": "General Physician"
    }

In [52]:
def triage_summary_agent(state: ClinicalState):

    prompt = f"""
    You are a Triage Summary Agent.

    Generate a concise triage summary including:
    - symptoms
    - severity
    - routed department
    - emergency status
    - patient history risk

    State:
    {state}
    """
    response = llm.invoke(prompt)
    return {
        "triage_summary": response.content.strip().lower()
    }

In [53]:
def specialist_suggestion_agent(state: ClinicalState):

    prompt = f"""
    You are a Specialist Suggestion Agent.

    Suggest the final specialist/department
    for patient consultation.

    State:
    {state}
    """
    response=llm.invoke(prompt)
    return {
        "specialist_suggestion": response.content.strip().lower()
    }

In [54]:
def diagnostic_test_agent(state: ClinicalState):

    if str(state.get("emergency_flag", "false")).lower() == "true":
        return {
            "diagnostic_tests": "defer non-urgent tests until emergency stabilization"
        }

    prompt = f"""
    You are a Diagnostic Test Recommendation Agent.

    Recommend appropriate diagnostic tests.

    State:
    {state}
    """
    response=llm.invoke(prompt)
    return {
        "diagnostic_tests": response.content.strip().lower()
    }

In [55]:
def emergency_indicator_agent(state: ClinicalState):

    prompt = f"""
    You are an Emergency Indicator Agent.

    Detect possible emergency conditions from symptoms, severity, and distress.
    Return either:
    - true
    - false

    Patient Input:
    {state['patient_input']}

    Symptoms:
    {state.get('symptoms')}

    Severity:
    {state.get('severity')}

    Distress:
    {state.get('distress_level')}
    """
    response = llm.invoke(prompt)
    return {
        "emergency_flag": response.content.strip().lower()
    }

In [56]:
def follow_up_instruction_agent(state: ClinicalState):

    if str(state.get("emergency_flag", "false")).lower() == "true":
        return {
            "follow_up": "emergency team to coordinate immediate follow-up after stabilization"
        }

    prompt = f"""
    You are a Follow-up Instruction Agent.

    Generate:
    - follow-up instructions
    - precautions
    - monitoring advice

    State:
    {state}
    """
    response=llm.invoke(prompt)
    return {
        "follow_up": response.content.strip().lower()
    }

In [57]:
def emergency_notification_agent(state: ClinicalState):

    if str(state.get("emergency_flag", "false")).lower() == "true":
        return {
            "emergency_notification": "emergency desk notified"
        }

    return {
        "emergency_notification": "notification not required"
    }

In [58]:
from langgraph.graph import StateGraph, END,START

graph = StateGraph(ClinicalState)

nodes = {

    # Stage 1
    "symptom_extraction_agent": symptom_extraction_agent,
    "risk_scoring_agent": risk_scoring_agent,
    "medical_entity_recognition_agent": medical_entity_recognition_agent,
    "distress_analysis_agent": distress_analysis_agent,
    "patient_history_analysis_agent": patient_history_analysis_agent,
    "emergency_indicator_agent": emergency_indicator_agent,

    # Stage 2
    "medical_guideline_retrieval_agent":
        medical_guideline_retrieval_agent,

    "similar_case_retrieval_agent":
        similar_case_retrieval_agent,

    "drug_interaction_retrieval_agent":
        drug_interaction_retrieval_agent,

    "specialist_recommendation_agent":
        specialist_recommendation_agent,

    # Stage 3
    "emergency_escalation_agent":
        emergency_escalation_agent,

    "cardiology_agent":
        cardiology_agent,

    "neurology_agent":
        neurology_agent,

    "pharmacy_review_agent":
        pharmacy_review_agent,

    "general_physician_agent":
        general_physician_agent,

    # Stage 4
    "triage_summary_agent":
        triage_summary_agent,

    "specialist_suggestion_agent":
        specialist_suggestion_agent,

    "diagnostic_test_agent":
        diagnostic_test_agent,

    "follow_up_instruction_agent":
        follow_up_instruction_agent,

    "emergency_notification_agent":
        emergency_notification_agent
}


for name, func in nodes.items():
    graph.add_node(name, func)


# Stage 1: symptom understanding starts in parallel where appropriate
graph.add_edge(START, "symptom_extraction_agent")
graph.add_edge(START, "distress_analysis_agent")
graph.add_edge(START, "patient_history_analysis_agent")

# Symptom extraction feeds downstream understanding
graph.add_edge("symptom_extraction_agent", "risk_scoring_agent")
graph.add_edge("symptom_extraction_agent", "medical_entity_recognition_agent")
graph.add_edge("risk_scoring_agent", "emergency_indicator_agent")

# Stage 2 and emergency handling
graph.add_conditional_edges(
    "emergency_indicator_agent",
    lambda state: "emergency_escalation_agent"
    if str(state.get("emergency_flag", "false")).lower() == "true"
    else "medical_guideline_retrieval_agent",
    {
        "medical_guideline_retrieval_agent":
            "medical_guideline_retrieval_agent",
        "emergency_escalation_agent":
            "emergency_escalation_agent"
    }
)

graph.add_conditional_edges(
    "medical_entity_recognition_agent",
    medication_route,
    {
        "drug_interaction_retrieval_agent":
            "drug_interaction_retrieval_agent",

        "similar_case_retrieval_agent":
            "similar_case_retrieval_agent"
    }
)

graph.add_edge(
    "medical_guideline_retrieval_agent",
    "specialist_recommendation_agent"
)

graph.add_edge(
    "similar_case_retrieval_agent",
    "specialist_recommendation_agent"
)

graph.add_edge(
    "drug_interaction_retrieval_agent",
    "specialist_recommendation_agent"
)

graph.add_edge(
    "patient_history_analysis_agent",
    "specialist_recommendation_agent"
)

graph.add_conditional_edges(
    "specialist_recommendation_agent",
    intelligent_routing,
    {
        "emergency_escalation_agent":
            "emergency_escalation_agent",

        "general_physician_agent":
            "general_physician_agent",

        "pharmacy_review_agent":
            "pharmacy_review_agent",

        "neurology_agent":
            "neurology_agent",

        "cardiology_agent":
            "cardiology_agent"
    }
)

routing_agents = [
    "general_physician_agent",
    "pharmacy_review_agent",
    "neurology_agent",
    "cardiology_agent"
]

care_agents = [
    "triage_summary_agent",
    "specialist_suggestion_agent",
    "diagnostic_test_agent",
    "follow_up_instruction_agent"
]

# Emergency path terminates early
graph.add_edge("emergency_escalation_agent", "emergency_notification_agent")
graph.add_edge("emergency_notification_agent", END)

# Normal care coordination in parallel
for route in routing_agents:
    for care in care_agents:
        graph.add_edge(route, care)

for care in care_agents:
    graph.add_edge(care, END)

workflow=graph.compile()

In [59]:
input_data={
    "patient_input":"I am having chest pain repeatedly and shortness of breath",
    "patient_history":"history of hypertension and heart disease"
}

In [ ]:
workflow.invoke(input_data)

{'patient_input': 'I am having chest pain repeatedly',
 'symptoms': '- chest pain repeatedly',
 'severity': '**severity classification: high**\n\n**reasoning:**\nchest pain is a significant symptom that can indicate a wide range of conditions, some of which are life-threatening (e.g., heart attack, pulmonary embolism, aortic dissection). the descriptor "repeatedly" increases the concern, suggesting an ongoing or recurring issue rather than a transient, benign event. given the potential for serious cardiac, pulmonary, or other systemic issues, immediate medical evaluation is crucial.\n\n**recommendation:**\n**seek immediate emergency medical attention. call your local emergency number (e.g., 911 in the us) or go to the nearest emergency room without delay.**',
 'medical_entities': '- **symptoms**: chest pain',
 'distress_level': 'high',
 'routed_agent': 'Emergency Escalation',
 'triage_summary': '**triage summary:**\n\n*   **symptoms:** chest pain repeatedly\n*   **severity:** high\n*  